# CS 281 Final Experiments — Is Chain-of-Thought Just Rationalization?

End-to-end pipeline for the final report. Runs on Colab/A100 (Llama-3-8B fp16).
Scale knobs live in the Config cell.

## 1. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
                    'transformer-lens>=2.0', 'datasets>=2.18',
                    'huggingface-hub>=0.22', 'accelerate>=0.28', 'seaborn'], check=True)

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv()
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN not set. Local: .env. Colab: Secret named HF_TOKEN.'
print('HF_TOKEN loaded:', os.environ['HF_TOKEN'][:6] + '...')

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.isdir('cot-faithfulness-audit'):
        subprocess.run(['git', 'clone', 'https://github.com/bballhaus/cot-faithfulness-audit.git'], check=True)
    os.chdir('/content/cot-faithfulness-audit')
sys.path.insert(0, os.path.abspath('.'))
print('cwd:', os.getcwd(), '| package present:', os.path.isdir('cot_faithfulness'))

In [ ]:
import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from cot_faithfulness import (data, model as model_mod, prompts, generation,
                              logit_lens, tuned_lens, patching, analysis, experiments)

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
torch.set_grad_enabled(False)
sns.set_context('notebook'); sns.set_style('whitegrid')

## 2. Config

In [ ]:
CFG = dict(
    model_name = model_mod.DEFAULT_MODEL,
    N_BOOLQ    = 1500,
    N_MNLI     = 1000,
    TAU        = 0.8,
    TAUS       = (0.5, 0.7, 0.8, 0.9),
    MAX_LEN    = 1024,
    COT_TOKENS = 180,
    TUNED_FIT_N = 96,
    TUNED_EVAL_N = 250,
    QUAL_K     = 25,
    SEED       = 42,
)
print(json.dumps(CFG, indent=2, default=str))

## 3. Load model

In [ ]:
model = model_mod.load_model(name=CFG['model_name'])
device = next(model.parameters()).device
N_LAYERS = model.cfg.n_layers
print(f'Loaded {model.cfg.model_name} on {device} | n_layers={N_LAYERS} | d_vocab={model.cfg.d_vocab}')

## 4. Load data

In [ ]:
boolq = data.load_boolq(n=CFG['N_BOOLQ'], seed=CFG['SEED'])
mnli  = data.load_mnli(n=CFG['N_MNLI'], seed=CFG['SEED'])
print('BoolQ:', len(boolq), '| MNLI:', len(mnli))

## 5. Commitment: no-CoT baseline vs with-CoT (BoolQ)

A rationalization account predicts the with-CoT commitment distribution shifts
earlier (shallower) than the no-CoT baseline.

In [ ]:
recs_nocot, probs_nocot = experiments.run_commitment(
    model, boolq, 'boolq', tau=CFG['TAU'], with_cot=False, max_len=CFG['MAX_LEN'])
pd.DataFrame(recs_nocot).to_csv(RESULTS / 'boolq_commitment_nocot.csv', index=False)
np.save(RESULTS / 'boolq_probs_nocot.npy', probs_nocot)
len(recs_nocot)

In [ ]:
recs_cot, probs_cot = experiments.run_commitment(
    model, boolq, 'boolq', tau=CFG['TAU'], with_cot=True,
    max_len=CFG['MAX_LEN'], max_new_tokens=CFG['COT_TOKENS'])
pd.DataFrame(recs_cot).to_csv(RESULTS / 'boolq_commitment_cot.csv', index=False)
np.save(RESULTS / 'boolq_probs_cot.npy', probs_cot)
len(recs_cot)

In [ ]:
summ_nocot = analysis.commitment_summary_ci(recs_nocot)
summ_cot   = analysis.commitment_summary_ci(recs_cot)
print('NO-COT  :', json.dumps(summ_nocot, indent=2, default=float))
print('WITH-COT:', json.dumps(summ_cot, indent=2, default=float))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
analysis.plot_commitment_compare(recs_nocot, recs_cot, N_LAYERS, ax=ax)
plt.tight_layout(); plt.savefig(RESULTS / 'commitment_compare.png', dpi=150); plt.show()

In [ ]:
def mean_pcorrect(probs, recs):
    labels = np.array([r['label'] for r in recs])
    return probs[np.arange(len(recs)), :, labels].mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mean_pcorrect(probs_nocot, recs_nocot), label='no-CoT')
ax.plot(mean_pcorrect(probs_cot, recs_cot), label='with-CoT')
ax.axhline(CFG['TAU'], color='red', ls='--', alpha=0.5, label=f"tau={CFG['TAU']}")
ax.set_xlabel('layer'); ax.set_ylabel('mean P(correct class)')
ax.set_title('Logit-lens confidence by depth: no-CoT vs with-CoT (BoolQ)')
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS / 'mean_curve_compare.png', dpi=150); plt.show()

## 6. Threshold sensitivity (tau sweep)

In [ ]:
labels_nocot = np.array([r['label'] for r in recs_nocot])
labels_cot   = np.array([r['label'] for r in recs_cot])
sweep_nocot = analysis.threshold_sweep(probs_nocot, labels_nocot, taus=CFG['TAUS'])
sweep_cot   = analysis.threshold_sweep(probs_cot,   labels_cot,   taus=CFG['TAUS'])
sweep_nocot['prompt'] = 'no-CoT'; sweep_cot['prompt'] = 'with-CoT'
sweep = pd.concat([sweep_nocot, sweep_cot], ignore_index=True)
sweep.to_csv(RESULTS / 'threshold_sweep.csv', index=False)
sweep

## 7. Qualitative extremes (with-CoT commitment)

In [ ]:
qual = analysis.qualitative_extremes(recs_cot, k=CFG['QUAL_K'], text_key='cot_text')
qual.to_csv(RESULTS / 'qualitative_extremes.csv', index=False)
print('early-commit example:\n', qual[qual.group=='early'].iloc[0]['cot_text'][:400])
print('\nlate-commit example:\n', qual[qual.group=='late'].iloc[0]['cot_text'][:400])
qual[['idx','commitment_layer','correct','group']]

## 8. Tuned-lens robustness comparison

Train per-layer affine translators on a calibration set, then recompute
commitment with both lenses on held-out examples.

In [ ]:
rng = random.Random(CFG['SEED'])
fit_idx = rng.sample(range(len(boolq)), min(CFG['TUNED_FIT_N'], len(boolq)))
fit_seqs = []
for i in fit_idx:
    ex = data.boolq_example(boolq[i])
    toks = model.to_tokens(prompts.format_boolq_direct(ex['passage'], ex['question']))
    if toks.shape[1] <= CFG['MAX_LEN']:
        fit_seqs.append(toks)
lens = tuned_lens.fit_tuned_lens(model, fit_seqs, rank=256, steps=250, lr=1e-3)
print('fitted tuned lens on', len(fit_seqs), 'calibration prompts')

In [ ]:
eval_idx = [i for i in range(len(boolq)) if i not in set(fit_idx)][:CFG['TUNED_EVAL_N']]
bt = prompts.boolq_target_tokens(model); tids = [bt['No'], bt['Yes']]
ll_layers, tl_layers = [], []
for i in tqdm(eval_idx, desc='tuned vs logit'):
    ex = data.boolq_example(boolq[i])
    toks = model.to_tokens(prompts.format_boolq_direct(ex['passage'], ex['question']))
    if toks.shape[1] > CFG['MAX_LEN']:
        continue
    pos = [toks.shape[1] - 1]
    p_ll = logit_lens.logit_lens(model, toks, tids, positions=pos)[:, 0, :]
    p_tl = tuned_lens.tuned_lens_probs(model, lens, toks, tids, positions=pos)[:, 0, :]
    ll_layers.append(logit_lens.commitment_layer(p_ll, ex['label'], CFG['TAU']))
    tl_layers.append(logit_lens.commitment_layer(p_tl, ex['label'], CFG['TAU']))
lens_cmp = {
    'logit_lens_mean_commit': analysis.bootstrap_ci([x for x in ll_layers if x >= 0]),
    'tuned_lens_mean_commit': analysis.bootstrap_ci([x for x in tl_layers if x >= 0]),
    'logit_frac_committed': float(np.mean(np.array(ll_layers) >= 0)),
    'tuned_frac_committed': float(np.mean(np.array(tl_layers) >= 0)),
}
print(json.dumps(lens_cmp, indent=2, default=float))
json.dump(lens_cmp, open(RESULTS / 'tuned_vs_logit.json', 'w'), indent=2, default=float)

## 9. CoT corruption + pre-CoT patching (BoolQ)

On MNLI, Llama-3 answered directly ~98% of the time (empty CoT -> dropped), leaving only n=25.
BoolQ produces a non-empty CoT 99.9% of the time, so we run corruption there and **reuse the
CoTs already generated in section 5** (`results/boolq_commitment_cot.csv`): no clean-CoT
regeneration. `random` needs no generation at all; only `invert` calls the model.

Requires a live session with `model` and `boolq` loaded — run sections 1-4 first
(you can skip 5-8). Then run the two cells below. Rough cost on an A100:
`random` (N=800) ~20 min, `invert` (N=300) ~45 min.

In [ ]:
import json
from pathlib import Path
import pandas as pd
R = Path('results')

_cot_df = pd.read_csv(R / 'boolq_commitment_cot.csv')
_cot_df = _cot_df[_cot_df['cot_text'].fillna('').astype(str).str.strip() != ''].reset_index(drop=True)
_bt = prompts.boolq_target_tokens(model); CHOICES = ['No', 'Yes']; TIDS = [_bt['No'], _bt['Yes']]

def run_boolq_corruption(strategy, n_max=None, seed_base=0, free_every=25):
    rows, skipped = [], 0
    sub = _cot_df if n_max is None else _cot_df.iloc[:n_max]
    for j, (_, r) in enumerate(tqdm(sub.iterrows(), total=len(sub), desc=f'boolq {strategy}')):
        idx = int(r['idx']); cot = str(r['cot_text'])
        ex = data.boolq_example(boolq[idx]); label = ex['label']
        prompt = prompts.format_boolq(ex['passage'], ex['question'])
        if model.to_tokens(prompt).shape[1] > CFG['MAX_LEN'] - 200:
            skipped += 1; continue
        res = patching.causal_corruption_test(
            model, prompt, cot, TIDS, strategy=strategy, seed=seed_base + idx, do_patch=True)
        rows.append({
            'idx': idx, 'label': label, 'label_text': CHOICES[label],
            'pred_clean': CHOICES[res['clean_argmax']], 'pred_corrupted': CHOICES[res['corrupted_argmax']],
            'flipped': bool(res['flipped']), 'logit_diff_drop': float(res['logit_diff_drop']),
            'correct_clean': res['clean_argmax'] == label, 'correct_corrupted': res['corrupted_argmax'] == label,
            'cot_len': int(res['cot_len']), 'pred_patched': CHOICES[res['patched_argmax']],
            'patch_recovers_clean': bool(res['patch_recovers_clean']),
            'correct_patched': res['patched_argmax'] == label,
        })
        if (j + 1) % free_every == 0:
            model_mod.free_memory()
    print(f'{strategy}: usable={len(rows)} skipped={skipped}')
    return rows

N_RANDOM = 800
boolq_random = run_boolq_corruption('random', n_max=N_RANDOM)
pd.DataFrame(boolq_random).to_csv(R / 'boolq_corruption_random.csv', index=False)
print(json.dumps(analysis.patching_summary(boolq_random, with_ci=True), indent=2, default=float))

In [ ]:
N_INVERT = 300
boolq_invert = run_boolq_corruption('invert', n_max=N_INVERT)
pd.DataFrame(boolq_invert).to_csv(R / 'boolq_corruption_invert.csv', index=False)

_summ = json.load(open(R / 'final_summary.json'))
_summ['boolq_corruption_random'] = analysis.patching_summary(boolq_random, with_ci=True)
_summ['boolq_corruption_invert'] = analysis.patching_summary(boolq_invert, with_ci=True)
json.dump(_summ, open(R / 'final_summary.json', 'w'), indent=2, default=float)
print(json.dumps(_summ['boolq_corruption_invert'], indent=2, default=float))

In [ ]:
sum_random = analysis.patching_summary(mnli_random, with_ci=True)
sum_invert = analysis.patching_summary(mnli_invert, with_ci=True)
print('RANDOM:', json.dumps(sum_random, indent=2, default=float))
print('INVERT:', json.dumps(sum_invert, indent=2, default=float))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, recs, name in [(axes[0], mnli_random, 'random'), (axes[1], mnli_invert, 'invert')]:
    df = pd.DataFrame(recs)
    sns.histplot(df['logit_diff_drop'], bins=30, ax=ax)
    fr = df['flipped'].mean()
    pr = df['patch_recovers_clean'].mean() if 'patch_recovers_clean' in df else float('nan')
    ax.set_title(f'{name}: flip={fr:.2f}, patch-recovery={pr:.2f}')
    ax.set_xlabel('clean P(ans) - corrupted P(ans)')
plt.tight_layout(); plt.savefig(RESULTS / 'mnli_corruption.png', dpi=150); plt.show()

## 10. Save final summary

In [ ]:
artifact = {
    'config': {k: str(v) for k, v in CFG.items()},
    'model_name': model.cfg.model_name,
    'n_layers': N_LAYERS,
    'boolq_commitment_nocot': summ_nocot,
    'boolq_commitment_cot': summ_cot,
    'threshold_sweep': sweep.to_dict(orient='records'),
    'tuned_vs_logit': lens_cmp,
    'mnli_corruption_random': sum_random,
    'mnli_corruption_invert': sum_invert,
}
json.dump(artifact, open(RESULTS / 'final_summary.json', 'w'), indent=2, default=float)
print('Saved', RESULTS / 'final_summary.json')

## 11. (Optional) Cross-family generalization

Re-run sections 3-5 with a second model to test whether late commitment is
Llama-specific. Alt models use different chat templates, so build prompts via
`prompts.format_chat` instead of the hand-built Llama-3 formatter.

## 12. Results analysis (reads saved `results/`, no model re-run)

Loads the saved artifacts and prints the quantitative summary + interpretation.
Safe to run on its own after a fresh runtime: it only touches files in `results/`.

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd

R = Path('results')
S = json.load(open(R / 'final_summary.json'))
nc, wc = S['boolq_commitment_nocot'], S['boolq_commitment_cot']
sweep = pd.DataFrame(S['threshold_sweep'])
tl = json.load(open(R / 'tuned_vs_logit.json')) if (R / 'tuned_vs_logit.json').exists() else S['tuned_vs_logit']
cr, ci_ = S['mnli_corruption_random'], S['mnli_corruption_invert']
N_MNLI = int(S['config']['N_MNLI'])
NL = S['n_layers']

def c(d): return f"{d['point']:.2f} [{d['lo']:.2f}, {d['hi']:.2f}]"

print('=' * 72)
print(f"MODEL: {S['model_name']} | {NL} layers")
print('=' * 72)

print("\n[1] BoolQ commitment depth (logit lens, tau=0.8, N=1500)")
print(f"    no-CoT   : layer {c(nc['mean_commitment_layer_ci'])}  acc {c(nc['accuracy_ci'])}")
print(f"    with-CoT : layer {c(wc['mean_commitment_layer_ci'])}  acc {c(wc['accuracy_ci'])}")
print(f"    -> CoT shifts commitment {wc['mean_commitment_layer']-nc['mean_commitment_layer']:+.2f} layers "
      f"({(wc['mean_commitment_layer']/(NL-1)):.0%} depth); acc delta {wc['accuracy']-nc['accuracy']:+.3f} (CIs overlap).")
print("    -> Under the logit lens the answer surfaces only in the last ~10% of layers, CoT or not.")

print("\n[2] Threshold robustness (mean depth fraction by tau)")
piv = sweep.pivot(index='tau', columns='prompt', values='mean_depth_frac')
print(piv.round(3).to_string())
print("    -> Deep commitment (0.90-0.99 depth) is robust to tau; not a threshold artifact.")

print("\n[3] Tuned lens vs logit lens (BoolQ, mean commitment layer)")
print(f"    logit lens : {c(tl['logit_lens_mean_commit'])}")
print(f"    tuned lens : {c(tl['tuned_lens_mean_commit'])}")
gap = tl['logit_lens_mean_commit']['point'] - tl['tuned_lens_mean_commit']['point']
print(f"    -> {gap:.1f}-layer gap. Answer is linearly DECODABLE ~{tl['tuned_lens_mean_commit']['point']:.0f}/{NL}"
      " once drift is corrected.")
print("    -> CAVEAT: tuned lens is a trained probe; 'decodable' != 'committed'. Report the gap, not layer 7 alone.")

if 'boolq_corruption_random' in S:
    rnd, inv, tag = S['boolq_corruption_random'], S['boolq_corruption_invert'], 'BoolQ'
else:
    rnd, inv, tag = cr, ci_, 'MNLI'
print(f"\n[4] CoT corruption + pre-CoT patching ({tag})")
for name, d in [('random-token', rnd), ('semantic-invert', inv)]:
    print(f"    {name:15s}: n={d['n']:>4d}  flip {c(d['flip_rate_ci'])}  "
          f"patch-recovery {c(d['patch_recovery_rate_ci'])}  clean-acc {d['accuracy_clean']:.2f}")
print(f"    -> invert flips {inv['flip_rate']/max(rnd['flip_rate'],1e-9):.1f}x more than random: CoT CONTENT is causal,")
print("       not just its tokens -> evidence against pure post-hoc rationalization.")
if tag == 'MNLI':
    print(f"    -> WARNING: only {cr['n']}/{N_MNLI} survived ({1-cr['n']/N_MNLI:.0%} skipped, empty CoT). Preliminary.")


## 13. Robustness improvements (paraphrase control, stat tests, lens control)

Four additions for the report: (1) a **paraphrase** corruption control (meaning preserved → flip should track *random*, not *invert*); (2) **statistical tests** on the flip-rate gaps; (3) a **tuned-lens validity control** (lens fit on shuffled targets) + per-layer accuracy; (5) **invert scaled to n=800** plus a **shuffle** baseline. Requires the model (§3), data (§4), and the tuned lens (§8) to be in memory.

In [ ]:
# 13a. Corruption runner; also stores the rewritten CoT text for provenance.
# do_patch=False: the pre-CoT patch is a no-op (recovery == 1-flip), so we skip it.
import json
from pathlib import Path
import pandas as pd
R = Path('results')

_cot = pd.read_csv(R / 'boolq_commitment_cot.csv')
_cot = _cot[_cot['cot_text'].fillna('').astype(str).str.strip() != ''].reset_index(drop=True)
_bt = prompts.boolq_target_tokens(model); CH = ['No', 'Yes']; TIDS = [_bt['No'], _bt['Yes']]

def run_corr(strategy, n_max=None, seed_base=0, free_every=25):
    rows, skipped = [], 0
    sub = _cot if n_max is None else _cot.iloc[:n_max]
    for j, (_, r) in enumerate(tqdm(sub.iterrows(), total=len(sub), desc=strategy)):
        idx = int(r['idx']); cot = str(r['cot_text'])
        ex = data.boolq_example(boolq[idx]); label = ex['label']
        prompt = prompts.format_boolq(ex['passage'], ex['question'])
        if model.to_tokens(prompt).shape[1] > CFG['MAX_LEN'] - 200:
            skipped += 1; continue
        res = patching.causal_corruption_test(
            model, prompt, cot, TIDS, strategy=strategy, seed=seed_base + idx, do_patch=False)
        rows.append({
            'idx': idx, 'label': label, 'label_text': CH[label],
            'pred_clean': CH[res['clean_argmax']], 'pred_corrupted': CH[res['corrupted_argmax']],
            'flipped': bool(res['flipped']), 'logit_diff_drop': float(res['logit_diff_drop']),
            'correct_clean': res['clean_argmax'] == label, 'correct_corrupted': res['corrupted_argmax'] == label,
            'cot_len': int(res['cot_len']), 'corrupted_cot_text': res.get('corrupted_cot_text'),
        })
        if (j + 1) % free_every == 0:
            model_mod.free_memory()
    print(f'{strategy}: usable={len(rows)} skipped={skipped}')
    return rows

In [ ]:
# 13b (#5). Scale invert to n=800 (matches random) and add a shuffle baseline (n=800).
boolq_invert800 = run_corr('invert', n_max=800)
pd.DataFrame(boolq_invert800).to_csv(R / 'boolq_corruption_invert800.csv', index=False)
boolq_shuffle = run_corr('shuffle', n_max=800)
pd.DataFrame(boolq_shuffle).to_csv(R / 'boolq_corruption_shuffle.csv', index=False)
for nm, rr in [('invert800', boolq_invert800), ('shuffle', boolq_shuffle)]:
    print(nm, json.dumps(analysis.patching_summary(rr, with_ci=True), default=float))

In [ ]:
# 13c (#1). Paraphrase control (n=800): meaning preserved -> flip should track random, not invert.
boolq_paraphrase = run_corr('paraphrase', n_max=800)
pd.DataFrame(boolq_paraphrase).to_csv(R / 'boolq_corruption_paraphrase.csv', index=False)
print(json.dumps(analysis.patching_summary(boolq_paraphrase, with_ci=True), indent=2, default=float))

In [ ]:
# 13d (#2). Statistical tests on flip-rate gaps (two-proportion z + bootstrap difference).
def _flips(name):
    return pd.read_csv(R / f'boolq_corruption_{name}.csv')['flipped'].astype(int).values
conds = {n: _flips(n) for n in ['random', 'shuffle', 'paraphrase', 'invert800']
         if (R / f'boolq_corruption_{n}.csv').exists()}
base = conds['random']; stat = {}
for name, fl in conds.items():
    if name == 'random': continue
    stat[f'{name}_vs_random'] = {
        'z': analysis.two_proportion_z(int(fl.sum()), len(fl), int(base.sum()), len(base)),
        'boot': analysis.bootstrap_diff(fl, base)}
if 'invert800' in conds and 'paraphrase' in conds:
    a, b = conds['invert800'], conds['paraphrase']
    stat['invert_vs_paraphrase'] = {
        'z': analysis.two_proportion_z(int(a.sum()), len(a), int(b.sum()), len(b)),
        'boot': analysis.bootstrap_diff(a, b)}
json.dump(stat, open(R / 'corruption_stats.json', 'w'), indent=2, default=float)
print(json.dumps(stat, indent=2, default=float))

In [ ]:
# 13e (#3). Tuned-lens validity control: refit a lens on SHUFFLED (resid, target) pairs.
# If the shuffled lens commits as early as the real one, early decodability is an artifact.
if 'fit_seqs' not in dir() or 'lens' not in dir():
    raise RuntimeError('Run section 8 (tuned lens) first to define fit_seqs, lens, eval_idx, tids.')
import numpy as np
lens_shuf = tuned_lens.fit_tuned_lens(model, fit_seqs, rank=256, steps=250, lr=1e-3, shuffle_targets=True)
tl_shuf_layers = []; acc_by_layer = np.zeros(model.cfg.n_layers); n_eval = 0
for i in tqdm(eval_idx, desc='lens control'):
    ex = data.boolq_example(boolq[i])
    toks = model.to_tokens(prompts.format_boolq_direct(ex['passage'], ex['question']))
    if toks.shape[1] > CFG['MAX_LEN']:
        continue
    pos = [toks.shape[1] - 1]
    p_shuf = tuned_lens.tuned_lens_probs(model, lens_shuf, toks, tids, positions=pos)[:, 0, :]
    p_real = tuned_lens.tuned_lens_probs(model, lens, toks, tids, positions=pos)[:, 0, :]
    tl_shuf_layers.append(logit_lens.commitment_layer(p_shuf, ex['label'], CFG['TAU']))
    acc_by_layer += (p_real.argmax(dim=1).numpy() == ex['label']); n_eval += 1
acc_by_layer /= max(n_eval, 1)
ctrl = {'tuned_shuffled_mean_commit': analysis.bootstrap_ci([x for x in tl_shuf_layers if x >= 0]),
        'tuned_shuffled_frac_committed': float(np.mean(np.array(tl_shuf_layers) >= 0)),
        'real_lens_acc_by_layer': acc_by_layer.tolist()}
json.dump(ctrl, open(R / 'tuned_lens_control.json', 'w'), indent=2, default=float)
print(json.dumps({k: v for k, v in ctrl.items() if k != 'real_lens_acc_by_layer'}, indent=2, default=float))
print('real lens acc by layer:', np.round(acc_by_layer, 2))